**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Optimization on Manifolds

When the constraint isn't a fence but a *surface* — unit spheres, orthonormal frames — penalties and projections fight the geometry. Riemannian optimization walks **along** the surface instead: two sessions, ending with an orthogonality-constrained eigenproblem solved natively and verified against `eigh`.

## 1. Pre-requisites

[Optimization](./Optimization.ipynb), [Linear Algebra](../Linear_Algebra/Linear_Algebra.ipynb) S3–S5.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)

---
### 🕐 Session 1 of 2 — *Riemannian Gradients & Retraction* (~40 min)
**Goal:** project the gradient onto the tangent space, step, retract — descent that never leaves the surface.
**Builds on:** [Optimization](./Optimization.ipynb) S2. &nbsp; **Feeds into:** Session 2 (the Stiefel manifold).

---

<details>
<summary>🎓 <b>Teacher notes — Session 1: Riemannian Gradients & Retraction</b></summary>

**Timing (~40 min).** 10 min why projection-and-penalty fight the geometry · 12 min the three-step recipe · 10 min the demo *and its failure* · 8 min what the failure teaches.

**⚠️ Read this first — the demo does not converge, and you must plan for it.** The committed output shows a final Rayleigh quotient of **0.5015** against a true minimum of **0.0097** (about 52× too high) and an eigenvector alignment of **0.53** where success would be ~1.0. The plot title claims it reaches "the bottom eigenvector, natively." It does not. Do not present this cell as a success; the debrief below treats it honestly, and there is a genuinely good lesson in the failure.

**Why it fails — and it is not the algorithm's fault.** `S = M @ M.T` with $M$ a $40\times40$ Gaussian is a Wishart matrix at $\gamma = p/n = 1$. By [Marchenko–Pastur](../Random_Matrix_Theory/Random_Matrix_Theory.ipynb), its density behaves like $1/\sqrt{x}$ near zero, so the *smallest* eigenvalues are densely packed. Rayleigh-quotient gradient descent converges linearly at a rate governed by the relative gap $(\lambda_2-\lambda_1)/(\lambda_{\max}-\lambda_1)$ — and with $\lambda_1,\lambda_2$ both near zero against a $\lambda_{\max}$ of order 160, that gap is minuscule. 300 iterations is nowhere near enough, and simply raising the count is a poor fix.

**Turn it into the session's best teaching moment.** Ask the room why Session 2 works perfectly (orthonormality drift 2.2e-16, principal angles exactly 1) while Session 1 does not, given both use the same three-step recipe. The answer is nothing to do with manifolds: Session 2 *ascends* toward the **top** eigenvectors, which in a Wishart matrix are well separated, so the gap is large and convergence is fast. Same algorithm, same manifold, opposite end of the spectrum, completely different conditioning. **Riemannian optimisation inherits ordinary gradient descent's conditioning problems — it fixes the constraint, not the curvature.** That is a more valuable lesson than a clean plot would have been.

**If you want a working demo**, the honest options are: ascend to the *largest* eigenvector instead (well-gapped, converges quickly); build `S` with a deliberately well-separated small eigenvalue rather than drawing Wishart; or add a Riemannian momentum/conjugate-gradient step. Do not simply raise `range(300)` — the gap is the problem, not the budget.

**The actual content of the session, which stands regardless.** The three-step dance — project the Euclidean gradient onto the tangent space, step, retract onto the manifold — is correct and elegant, and Session 2 verifies it works. The key algebraic fact is worth putting on the board: on the sphere with $f = x^\top S x$, the tangent-projected gradient is $2(Sx - (x^\top Sx)x)$, which vanishes **exactly at eigenvectors**. Eigenproblems *are* Riemannian critical points, which is a genuinely satisfying reframing of a familiar object.

**Ask the room.** "Why not just add a penalty $\lambda(\|x\|^2-1)^2$ and use ordinary gradient descent?" Because you then trade one problem for two: tuning $\lambda$, and accepting that the iterate is never exactly on the manifold. Retraction gives exact feasibility at every step for free — Session 2's 2.2e-16 drift is that claim, measured.
</details>

## 2. Walking on Curved Ground

💡 **Intuition.** On a sphere, the Euclidean gradient points *off* the surface — following it and re-normalizing is a fight. The Riemannian recipe makes peace with the geometry: (1) **project** the gradient onto the tangent plane (the directions you can actually move), (2) step, (3) **retract** back onto the manifold (for the sphere: normalize). All the [convergence theory](./Optimization.ipynb) carries over with Euclidean distance replaced by geodesic distance. On the sphere with $f = x^T S x$, the tangent-projected gradient is $2(Sx - (x^TSx)x)$ — zero exactly at **eigenvectors**: eigenproblems ARE Riemannian critical points (as [Lagrange already hinted](./Optimization.ipynb)).

In [2]:
# Rayleigh-quotient minimization on the sphere — ORACLE: numpy's eigh
n_dim = 40
M = rng.standard_normal((n_dim, n_dim)); S = M @ M.T
w_true, V_true = np.linalg.eigh(S)

x = rng.standard_normal(n_dim); x /= np.linalg.norm(x)
vals = []
eta = 0.5 / np.abs(w_true).max()
for it in range(300):
    egrad = 2 * S @ x
    rgrad = egrad - (x @ egrad) * x            # project onto tangent space  T_x = {v : xᵀv = 0}
    x = x - eta * rgrad
    x /= np.linalg.norm(x)                     # retract to the sphere
    vals.append(x @ S @ x)

print(f"Riemannian GD:  final Rayleigh quotient {vals[-1]:.8f}")
print(f"eigh oracle:    smallest eigenvalue     {w_true[0]:.8f}")
print(f"eigenvector alignment |⟨x, v_min⟩| = {abs(x @ V_true[:, 0]):.6f}")
plt.figure(figsize=(7, 2.4))
plt.semilogy(np.array(vals) - w_true[0])
plt.title("descent on the sphere → the bottom eigenvector, natively")
plt.xlabel("iteration"); plt.ylabel("gap to λ_min"); plt.tight_layout(); plt.show()

Riemannian GD:  final Rayleigh quotient 0.50152742
eigh oracle:    smallest eigenvalue     0.00972568
eigenvector alignment |⟨x, v_min⟩| = 0.532710


/tmp/ipykernel_2982305/483961448.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.xlabel("iteration"); plt.ylabel("gap to λ_min"); plt.tight_layout(); plt.show()


**What just happened — this run did not converge, and the plot title overstates it.** Read the three printed numbers against each other:

| | value |
|---|---|
| final Rayleigh quotient | **0.5015** |
| true smallest eigenvalue | **0.0097** |
| eigenvector alignment $\lvert\langle x, v_{\min}\rangle\rvert$ | **0.533** |

The quotient is about **52× too high**, and an alignment of 0.53 means the iterate is roughly 58° away from the target eigenvector. Successful convergence would show a quotient matching `eigh` to several decimals and an alignment near 1.000. The title's claim that descent reaches "the bottom eigenvector, natively" is not what this run achieved.

**And the cause is not the Riemannian machinery — it is conditioning.** `S = M @ M.T` with $M$ a $40\times40$ Gaussian is a Wishart matrix at $\gamma = p/n = 1$. From [Marchenko–Pastur](../Random_Matrix_Theory/Random_Matrix_Theory.ipynb), its eigenvalue density behaves like $1/\sqrt{x}$ as $x \to 0$, so the smallest eigenvalues are **densely clustered near zero**. Rayleigh-quotient gradient descent converges linearly at a rate set by the *relative gap*
$$\frac{\lambda_2 - \lambda_1}{\lambda_{\max} - \lambda_1},$$
and here $\lambda_1$ and $\lambda_2$ are both essentially zero while $\lambda_{\max}$ is of order 160. That gap is tiny, so convergence is glacial and 300 iterations is nowhere near sufficient.

**The instructive comparison is with Session 2, which works perfectly.** Same three-step recipe, same manifold family — orthonormality drift 2.2e-16, principal-angle cosines exactly 1.0, trace matched to four decimals. The difference is that Session 2 *ascends* toward the **largest** eigenvalues, which in a Wishart matrix are well separated. Large gap, fast convergence. Small gap, no convergence.

So the lesson is sharper than a clean plot would have given: **Riemannian optimisation fixes the constraint, not the conditioning.** Working on a manifold guarantees you stay feasible at every iterate; it inherits every convergence difficulty ordinary gradient descent has. If your problem is ill-conditioned in Euclidean space, it is ill-conditioned on the manifold too.

**If you want this cell to succeed**, the fix is not more iterations. Either ascend to the top eigenvector (well-gapped), construct `S` with a deliberately isolated small eigenvalue instead of drawing Wishart, or add Riemannian momentum or conjugate gradients. Raising `range(300)` alone chases a rate that is fundamentally slow.

**What the session still establishes correctly.** The three-step dance is right: project the Euclidean gradient onto the tangent space, step, retract. And the key algebraic fact holds regardless of whether this run converged — on the sphere with $f = x^\top Sx$, the tangent-projected gradient $2(Sx - (x^\top Sx)x)$ vanishes **exactly at eigenvectors**. Eigenproblems are Riemannian critical points. Session 2 verifies the machinery against `eigh` under conditions where the optimisation can actually finish.

---
### 🕐 Session 2 of 2 — *The Stiefel Manifold: Orthonormal Frames* (~40 min)
**Goal:** optimize over ORTHONORMAL MATRICES; recover a subspace, verified against eigh.
**Builds on:** Session 1.

---

<details>
<summary>🎓 <b>Teacher notes — Session 2: The Stiefel Manifold</b></summary>

**Timing (~40 min).** 8 min what a frame is and why constraints get harder · 10 min the tangent projection · 10 min QR as retraction · 12 min the demo and the comparison with Session 1.

**Open by scaling up the constraint.** Session 1 constrained one vector to the sphere. Now constrain a whole $n \times k$ matrix so that $X^\top X = I$ — an orthonormal *frame*. Ask how many scalar constraints that is: $k(k+1)/2$, so 15 for $k = 5$. A Lagrange-multiplier treatment would carry 15 multipliers and still only enforce orthonormality approximately. The Stiefel manifold makes all 15 automatic.

**Point at where each constraint type appears**, because the abstraction lands better with examples: PCA subspaces, [dictionary](../../Intro_DSP/Sparse_Dictionary_Learning.ipynb) atoms, [beamformer banks](../../Intro_DSP/Array_Processing.ipynb), and ICA's whitened rotations from [BSS](../../Intro_DSP/ICA_Blind_Source_Separation.ipynb). Orthonormal frames are not an exotic constraint — they are most of applied linear algebra.

**QR as the retraction is the elegant bit.** The `Q` factor of a QR decomposition *is* the nearest orthonormal frame, so "step off the manifold, then take Q" is a principled projection rather than a hack. Students who did [Numerical Linear Algebra](../Numerical_Linear_Algebra/Numerical_Linear_Algebra.ipynb) already know QR is numerically excellent, which is exactly why the drift comes out at machine precision rather than merely small.

**Make the drift number the headline.** $\|X^\top X - I\| = 2.2\times10^{-16}$ after 500 iterations — one unit in the last place, with **no accumulation at all**. Contrast explicitly with a penalty method, where the iterate is never exactly feasible and the violation grows or oscillates depending on the penalty weight. Ask what a controls or robotics engineer would prefer: a rotation matrix that is orthonormal to machine precision at every step, or one that is approximately orthonormal and drifting.

**The verification is stronger than it looks — explain principal angles.** Comparing subspaces is not comparing matrices: two different orthonormal bases can span the same subspace, so `X` need not equal `V_top`. The singular values of $X^\top V_{\text{top}}$ are the cosines of the principal angles between the two subspaces, and all five coming back as exactly 1.0 means the subspaces coincide, whatever basis each uses. That is the right invariant to test, and it is worth a minute — students often try to compare bases directly and get confused when a correct answer looks wrong.

**Use the contrast with Session 1 deliberately.** Same recipe, same manifold family, and this one converges perfectly while Session 1 does not. The reason is spectral gap, not geometry: the *top* eigenvalues of a Wishart matrix are well separated, the bottom ones are densely packed. Having both a success and a failure in one workshop is genuinely valuable — it shows that Riemannian optimisation fixes feasibility and inherits conditioning, which is the honest summary of the method.

**Note the practical payoff mentioned in the closing cell.** Orthogonality-constrained RNNs kill the [vanishing gradient](../../Intro_Time_Series/Intro_RNN.ipynb) analytically, because a unitary recurrence has all singular values equal to 1 — no decay, no explosion. That is this manifold solving a problem the RNN workshop could only mitigate.
</details>

## 3. Frames That Stay Frames

💡 **Intuition.** Many problems want a whole orthonormal *frame* $X \in \mathbb{R}^{n \times k}$, $X^TX = I$ — PCA subspaces, [dictionary](../../Intro_DSP/Sparse_Dictionary_Learning.ipynb) atoms, [beamformer banks](../../Intro_DSP/Array_Processing.ipynb). That set is the **Stiefel manifold**. Same three-step dance: tangent projection $\xi = G - X\,\mathrm{sym}(X^TG)$, step, retract via the [QR factorization](../Numerical_Linear_Algebra/Numerical_Linear_Algebra.ipynb) — Q *is* the nearest-frame map. No Lagrange multipliers, no drift, orthonormal to machine precision at every iterate.

In [3]:
# top-k subspace by Stiefel gradient ASCENT on tr(XᵀSX) — ORACLE: eigh's top-k subspace
k = 5
X = np.linalg.qr(rng.standard_normal((n_dim, k)))[0]
eta = 0.5 / np.abs(w_true).max()
for it in range(500):
    G = 2 * S @ X
    sym = (X.T @ G + G.T @ X) / 2
    xi = G - X @ sym                            # tangent projection
    X, _ = np.linalg.qr(X + eta * xi)           # step + QR retraction

V_top = V_true[:, -k:]
# subspace distance: principal angles via SVD of the cross-Gram
sv = np.linalg.svd(X.T @ V_top, compute_uv=False)
print(f"orthonormality drift ‖XᵀX − I‖ = {np.abs(X.T @ X - np.eye(k)).max():.2e}")
print(f"principal-angle cosines vs eigh's top-{k} subspace: {sv.round(6)}")
print(f"trace captured: {np.trace(X.T @ S @ X):.4f}  vs optimal {w_true[-k:].sum():.4f}")
assert sv.min() > 0.9999

orthonormality drift ‖XᵀX − I‖ = 2.22e-16
principal-angle cosines vs eigh's top-5 subspace: [1. 1. 1. 1. 1.]
trace captured: 620.8893  vs optimal 620.8893


**What just happened.** Three results, and all three are exact:

- Orthonormality drift $\|X^\top X - I\| = 2.2\times10^{-16}$ — one unit in the last place of double precision, after **500 iterations**, with no accumulation whatsoever.
- Principal-angle cosines against `eigh`'s top-5 subspace: `[1. 1. 1. 1. 1.]`, with the `assert` requiring every one above 0.9999.
- Trace captured 620.8893 against an optimal 620.8893.

**The drift figure is the claim worth making loudly.** We enforced $k(k+1)/2 = 15$ scalar constraints, and after 500 steps they hold to machine precision — not approximately, not with a tuned penalty weight, and with no tendency to grow. A penalty method $\lambda\|X^\top X - I\|^2$ would need $\lambda$ tuned, would never be exactly feasible, and would drift or oscillate depending on the weight. Retraction gives exact feasibility *for free at every iterate*, which is why this matters in applications where a matrix must genuinely be a rotation — robotics, attitude estimation, unitary recurrent networks.

QR is what makes it clean: the `Q` factor is the **nearest orthonormal frame** to a given matrix, so "step off the manifold, then take Q" is a principled projection rather than a repair hack. And it is numerically excellent, which is why the drift is machine-epsilon rather than merely small.

**The principal-angle check is the right verification, and it is subtler than it looks.** We are recovering a *subspace*, not a specific basis — two different orthonormal bases can span the same space, so demanding $X = V_{\text{top}}$ would be wrong and would fail on a correct answer. The singular values of $X^\top V_{\text{top}}$ are the cosines of the principal angles between the two subspaces, and all five equalling 1.0 means the subspaces **coincide exactly**, whatever bases they happen to use. Testing the invariant rather than the representation is the general lesson.

**Now compare with Session 1, because the contrast is the most useful thing here.** Same three-step recipe, same manifold family, and this one lands exactly while Session 1 finished 52× away from its target. The difference is not geometry — it is **spectral gap**. Session 2 ascends toward the *largest* eigenvalues of a Wishart matrix, which are well separated, so convergence is fast. Session 1 descends toward the *smallest*, which are densely clustered near zero, so convergence is glacial.

Having both in one workshop is more instructive than two successes would have been: **Riemannian optimisation guarantees feasibility and inherits conditioning.** Staying on the manifold is free; converging on it is exactly as hard as the underlying problem.

**And one payoff worth naming.** Optimising over orthonormal matrices is how orthogonality-constrained RNNs eliminate the [vanishing gradient](../../Intro_Time_Series/Intro_RNN.ipynb) *analytically* — a unitary recurrence has every singular value equal to 1, so gradients neither decay nor explode by construction. That is this manifold solving, at the level of structure, a problem the RNN workshop could only mitigate with gates and clipping.

**Where this bites in practice:** orthogonality-regularized RNNs (unitary evolution kills the [vanishing gradient](../../Intro_Time_Series/Intro_RNN.ipynb) analytically), ICA's whitened rotations ([BSS workshop](../../Intro_DSP/ICA_Blind_Source_Separation.ipynb)), and low-rank matrix completion on fixed-rank manifolds.

## 4. Conclusion

Project to the tangent, step, retract: constrained optimization without constraints, converging to `eigh`'s answers (verified to 6 decimals) while staying orthonormal to machine precision. When your parameter *is* a geometry, optimize in it.

---
## Where next

- [Numerical Linear Algebra](../Numerical_Linear_Algebra/Numerical_Linear_Algebra.ipynb) — QR as the retraction workhorse.
- [Sparse & Dictionary Learning](../../Intro_DSP/Sparse_Dictionary_Learning.ipynb) — unit-norm atom constraints, everywhere.